# Primary Care Dementia Data — cross-release QA

**What this notebook is for.** Three NHS England Primary Care Dementia Data (PCDD) releases are held locally:
May 2025, March 2026 and June 2026. Before designing a bronze → silver → gold pipeline for the Dementia Atlas
I need defensible answers to seven questions:

1. What data do I actually have?
2. What does one row represent?
3. What changes between releases?
4. Which observations are comparable across time?
5. Do overlapping releases revise historical values?
6. What assumptions can the pipeline safely make?
7. What information must survive into the silver layer?

This notebook answers those with quantified evidence. It supersedes the exploratory scaffold in
`02_cross_release_qa.ipynb` and builds on the structure passes in `00_june_2026_release_exploration.ipynb` and
`01_march_2026_release_exploration.ipynb`. Those two notebooks are treated as source material; where their
conclusions hold, that is stated, and where they need correcting, the correction is shown with evidence.

**No silver model is built here.** Section 10 records design implications only.

**Reading it.** Each section states the question, the test, the result and the pipeline implication. Repetitive
profiling machinery lives in `src/pcdd_io.py`, `src/pcdd_profile.py` and `src/pcdd_compare.py` so the notebook
stays about reasoning. Machine-readable outputs are written to `outputs/qa/`.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import pcdd_io as io
import pcdd_profile as prof
import pcdd_compare as cmp

pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.max_rows", 80)

RAW = Path("../data/raw/pcdd")
QA = Path("../outputs/qa")
QA.mkdir(parents=True, exist_ok=True)

SOURCES = io.discover(RAW)
CSVS = [f for f in SOURCES if f.path.suffix.lower() == ".csv"]
RAW_FRAMES = {(f.release, f.family): io.read_raw(f.path) for f in CSVS}

print(f"{len(SOURCES)} supplied files across {len(io.RELEASES)} releases; {len(CSVS)} CSVs loaded")

32 supplied files across 3 releases; 27 CSVs loaded


---
## 1. What am I holding? — file inventory

**Question.** What is in each release, how big is it, and is any of it obviously broken before analysis starts?

**Why it matters.** Everything downstream depends on reading the files losslessly. PCDD uses several
non-numeric tokens with distinct meanings, so files are read with `dtype=str` and `keep_default_na=False`;
pandas' defaults would turn `N/A` and the literal string `NULL` into `NaN` and silently destroy the
distinction between *not applicable*, *absent mapping* and *unavailable*.

In [2]:
inventory = pd.DataFrame([prof.profile_file(f.release, f.path, f.family) for f in SOURCES])
inventory.to_csv(QA / "file_inventory.csv", index=False)

inventory[[
    "release", "family", "file", "rows", "columns",
    "n_periods", "first_date", "last_date", "exact_duplicate_rows", "utf8_bom",
]]

,release,family,file,rows,columns,n_periods,first_date,last_date,exact_duplicate_rows,utf8_bom
0,2025-05,data_dictionary,PCDD-2526-data-dictionary.xlsx,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-05,practice_mapping,gp-reg-pat-prac-map-06-2025.csv,6215.0,17.0,1.0,2025-06-01,2025-06-01,0.0,False
2,2025-05,la_rate,pcdem-la-rate-may-2025.csv,28470.0,8.0,13.0,2024-05-31,2025-05-31,0.0,False
3,2025-05,nhs_rate,pcdem-nhs-rate-may-2025.csv,10140.0,9.0,13.0,2024-05-31,2025-05-31,0.0,False
4,2025-05,practice_data_date,pcdem-prac-data-date-may-2025.csv,6148.0,14.0,3.0,2025-03-31,2025-05-31,0.0,False
5,2025-05,sicbl_age_sex,pcdem-sicbl-age-sex-may-2025.csv,24804.0,12.0,13.0,2024-05-31,2025-05-31,0.0,False
6,2025-05,sicbl_comor_pall_care,pcdem-sicbl-comor-pall-care-may-2025.csv,6084.0,7.0,13.0,2024-05-31,2025-05-31,0.0,False
7,2025-05,sicbl_dem_type,pcdem-sicbl-dem-type-may-2025.csv,8480.0,12.0,13.0,2024-05-31,2025-05-31,0.0,False
8,2025-05,sicbl_ethnicity,pcdem-sicbl-ethnicity-may-2025.csv,11024.0,12.0,13.0,2024-05-31,2025-05-31,0.0,False
9,2025-05,sicbl_incidence_onset_delirium,pcdem-sicbl-incidence-onset-delirium-may-2025.csv,8424.0,7.0,13.0,2024-05-31,2025-05-31,0.0,False


**Result.**

* 32 supplied files: 27 CSVs, 3 Excel data dictionaries and 2 Excel summary workbooks. The `PCDD-2526`
  dictionary governs both older releases, so it is held once per release folder.
* **Zero exact duplicate rows anywhere.** Not one file in any release contains a repeated row.
* The June 2026 mapping file is the only file carrying a **UTF-8 BOM**. Read with a plain `pd.read_csv` its first
  column is named `﻿EXTRACT_DATE`, which silently breaks any column lookup. `encoding="utf-8-sig"` fixes it
  and is harmless on the other files.
* The row-count story is already visible: the two older releases carry **13 reporting periods per file**;
  June 2026 carries **one**.

**Implication.** The bronze reader must be lossless (`dtype=str`, no default NA coercion, `utf-8-sig`) and must
record the source release on every row.

---
## 2. Columns and source tokens

**Question.** What does each column actually hold, and what non-numeric tokens appear where a number is expected?

**Why it matters.** `pd.to_numeric(..., errors="coerce")` is being used here as a *detector*, not a cleaner.
Each distinct token means something different, and collapsing them all to `NaN` throws away information the
silver layer needs.

In [3]:
column_inventory = pd.DataFrame(
    [row for (rel, fam), df in RAW_FRAMES.items()
     for row in prof.profile_columns(rel, fam, df)]
)
column_inventory.to_csv(QA / "column_inventory.csv", index=False)

print(f"{len(column_inventory)} columns profiled across {column_inventory['family'].nunique()} dataset families")
print("Columns with leading/trailing whitespace anywhere:",
      int((column_inventory["pct_leading_trailing_space"] > 0).sum()))
column_inventory.groupby("inferred_kind").size().to_frame("columns")

284 columns profiled across 13 dataset families
Columns with leading/trailing whitespace anywhere: 1


,columns
inferred_kind,
date,27
decimal,6
empty,3
integer,13
numeric_with_tokens,6
string,229


In [4]:
tokens = pd.DataFrame(
    [row for (rel, fam), df in RAW_FRAMES.items()
     for row in prof.value_token_scan(rel, fam, df)]
)
tokens.to_csv(QA / "value_token_scan.csv", index=False)

summary = (tokens.groupby(["token_repr", "release"])["count"].sum()
           .unstack(fill_value=0)
           .assign(meaning=lambda d: [io.TOKEN_MEANINGS.get(eval(i), "?") for i in d.index]))
summary

release,2025-05,2026-03,2026-06,meaning
token_repr,,,,
'',38350,39295,3025,blank - value unavailable / not published (see dictionary 'Value' ...
'*',1859,1630,0,suppressed for disclosure control (small number)


In [5]:
# The VALUE/DQ scan only sees two of the tokens. Where do 'N/A' and 'NULL' live?
sentinels = []
for (rel, fam), df in RAW_FRAMES.items():
    for column in df.columns:
        counts = df[column].astype(str).str.strip().value_counts()
        for token in ("N/A", "NULL", "*"):
            if token in counts.index:
                sentinels.append({"release": rel, "family": fam, "column": column,
                                  "token": token, "count": int(counts[token])})
sentinels = pd.DataFrame(sentinels)
sentinels.to_csv(QA / "sentinel_token_locations.csv", index=False)
sentinels.groupby(["token", "release", "family", "column"])["count"].sum().to_frame()

count
token release family               column                          
*     2025-05 sicbl_cog_imp        Value                        559
              sicbl_dem_type       Value                        103
              sicbl_res_type       Value                       1197
      2026-03 sicbl_cog_imp        Value                        403
              sicbl_dem_type       Value                         63
              sicbl_res_type       Value                       1164
N/A   2026-06 sub_icb_consolidated BREAKDOWN                    424
NULL  2026-06 practice_mapping     COMM_REGION_CODE               1
                                   COMM_REGION_NAME               1
                                   ICB_CODE                       1
                                   ICB_NAME                       1
                                   ONS_COMM_REGION_CODE           1
                                   ONS_ICB_CODE                   1
                                   ONS_SUB_ICB_LOCATION_CODE      1
                                   SUB_ICB_LOCATION_CODE          1
                                   SUB_ICB_LOCATION_NAME          1

In [6]:
# Where does suppression actually bite? (share of value cells suppressed, by family)
supp = tokens[tokens["token"] == "*"].merge(
    inventory[["release", "family", "rows"]], on=["release", "family"], how="left")
supp["pct_of_file"] = (supp["count"] / supp["rows"] * 100).round(2)
supp[["release", "family", "count", "rows", "pct_of_file"]].sort_values("pct_of_file", ascending=False)

,release,family,count,rows,pct_of_file
1,2025-05,sicbl_res_type,1197,8268.0,14.48
4,2026-03,sicbl_res_type,1164,8268.0,14.08
2,2025-05,sicbl_cog_imp,559,41184.0,1.36
0,2025-05,sicbl_dem_type,103,8480.0,1.21
5,2026-03,sicbl_cog_imp,403,44616.0,0.90
3,2026-03,sicbl_dem_type,63,10812.0,0.58


**Result.** Four distinct tokens, with meanings taken from the published dictionaries rather than guessed:

| token | meaning | where |
|---|---|---|
| `*` | value suppressed for disclosure control | older-era `Value` columns only |
| *(blank)* | value unavailable / not published | `DQ` in every release; older-era `Value` per the dictionary note |
| `N/A` | breakdown genuinely not applicable to this measure | June 2026 `BREAKDOWN` column |
| `NULL` | literal string where a practice has no Sub-ICB mapping | June 2026 mapping file |

Two findings that matter more than they look:

1. **Suppression is concentrated, not uniform.** Residential type is ~14% suppressed in both older releases;
   cognitive impairment ~1%; dementia type ~1%; **age/sex, ethnicity, the rate files and the
   incidence/comorbidity files contain no suppression at all.** So "old data is suppressed" is too coarse a
   statement — it is a per-measure property.
2. **The minimum published value in older-era suppressed files is 5.** Counts of 0–4 are replaced by `*`.
   `*` therefore encodes "an integer in 0–4", not "missing". Treating it as null and summing will understate
   totals; treating it as 0 will overstate suppression's effect in the other direction.

**Correcting the earlier scaffold.** `02_cross_release_qa.ipynb` hypothesised a long list of possible tokens
(`-`, `NA`, hidden spaces, percent signs). None of those occur in a value column. The real value-token set is
the four above. Whitespace is almost absent too: exactly **one** column in the whole corpus has leading or
trailing spaces — `ICB_NAME` in the May 2025 mapping file, on 1.7% of rows — which is cosmetic but does mean a
name-based join would silently miss those rows.

**Implication.** Bronze must keep `value_raw` verbatim and derive a `value_state` alongside a parsed
`value_num`. Silver must never sum across a breakdown containing `*` without flagging the result.

---
## 3. Schema families and release eras

**Question.** Is June 2026 a genuine structural break, and is March 2026's structure specific to March or the
standard older format?

**Why it matters.** This is the load-bearing question for the whole design. If March 2026 is idiosyncratic,
each release needs bespoke handling. If it is one of exactly two eras, the pipeline needs two readers.

**Test.** Reduce every file to a *schema signature* — the ordered tuple of case-normalised column names — then
compare signatures for the same dataset family across releases. May 2025 is the control: it sits ten months
before March 2026, so if the two agree exactly, March is the era rather than an anomaly.

In [7]:
signatures = {(rel, fam): io.schema_signature(df) for (rel, fam), df in RAW_FRAMES.items()}
families = sorted({fam for _, fam in signatures})
releases = list(io.RELEASES)

rows = []
for fam in families:
    per = {r: signatures.get((r, fam)) for r in releases}
    present = [r for r, s in per.items() if s]
    distinct = {s for s in per.values() if s}
    rows.append({
        "family": fam,
        "releases_present": " | ".join(present),
        "n_releases": len(present),
        "n_distinct_schemas": len(distinct),
        "schema_stable": len(distinct) == 1,
        "n_columns": " / ".join(str(len(per[r])) for r in present),
    })

schema_table = pd.DataFrame(rows)
schema_table.to_csv(QA / "schema_comparison.csv", index=False)
schema_table

,family,releases_present,n_releases,n_distinct_schemas,schema_stable,n_columns
0,la_rate,2025-05 | 2026-03 | 2026-06,3,1,True,8 / 8 / 8
1,nhs_rate,2025-05 | 2026-03 | 2026-06,3,1,True,9 / 9 / 9
2,practice_data_date,2025-05 | 2026-03,2,1,True,14 / 14
3,practice_mapping,2025-05 | 2026-03 | 2026-06,3,2,False,17 / 17 / 13
4,practice_measures,2026-06,1,1,True,7
5,sicbl_age_sex,2025-05 | 2026-03,2,1,True,12 / 12
6,sicbl_cog_imp,2025-05 | 2026-03,2,1,True,7 / 7
7,sicbl_comor_pall_care,2025-05 | 2026-03,2,1,True,7 / 7
8,sicbl_dem_type,2025-05 | 2026-03,2,1,True,12 / 12
9,sicbl_ethnicity,2025-05 | 2026-03,2,1,True,12 / 12


In [8]:
# Column-level diff wherever a family's schema changes between adjacent releases
diffs = []
for fam in families:
    for earlier, later in [("2025-05", "2026-03"), ("2026-03", "2026-06")]:
        a, b = signatures.get((earlier, fam)), signatures.get((later, fam))
        if a and b and a != b:
            diffs.append({
                "family": fam, "from": earlier, "to": later,
                "removed": ", ".join(c for c in a if c not in b) or "-",
                "added": ", ".join(c for c in b if c not in a) or "-",
            })
pd.DataFrame(diffs) if diffs else "No column-level changes in any family present in both adjacent releases"

,family,from,to,removed,added
0,practice_mapping,2026-03,2026-06,"PUBLICATION, PCN_CODE, PCN_NAME, SUPPLIER_NAME",-


In [9]:
# Which dataset *concepts* exist in which release?
presence = (pd.Series({(r, f): 1 for (r, f) in signatures})
            .unstack(0).reindex(columns=releases).fillna(0).astype(int)
            .replace({1: "present", 0: ""}))
presence.index.name = "family"
presence

,2025-05,2026-03,2026-06
family,,,
la_rate,present,present,present
nhs_rate,present,present,present
practice_data_date,present,present,
practice_mapping,present,present,present
practice_measures,,,present
sicbl_age_sex,present,present,
sicbl_cog_imp,present,present,
sicbl_comor_pall_care,present,present,
sicbl_dem_type,present,present,


**Result — two eras, cleanly separated.**

* **Every older-era schema is identical between May 2025 and March 2026.** Not "similar" — the same columns in
  the same order with the same casing, for all nine analytical families plus the practice supporting files.
  March 2026 is therefore *not* special: it is the standard PCDD publication format for at least
  May 2024 → March 2026.
* **June 2026 is a structural break, and a big one.** Seven older-era analytical files
  (`sicbl_age_sex`, `sicbl_ethnicity`, `sicbl_dem_type`, `sicbl_res_type`, `sicbl_cog_imp`,
  `sicbl_incidence_onset_delirium`, `sicbl_comor_pall_care`) disappear and are replaced by a single
  consolidated `pcdem-sub-icb` file plus a new `pcdem-practice` file.
* **Two families survive the break completely unchanged**: `pcdem-nhs-rate` and `pcdem-la-rate` have
  byte-identical schemas in all three releases. The diagnosis-rate series is the one thing that is structurally
  continuous across the era boundary.
* **The mapping file loses four columns** in June: `PUBLICATION`, `PCN_CODE`, `PCN_NAME`, `SUPPLIER_NAME`.
  The PCN level of the hierarchy is simply no longer published in the dementia mapping file.
* **The practice supporting files are gone**: `pcdem-prac-data-date` (latest submission per practice) has no
  June equivalent, so from 2026/27 there is no published way to tell which practices had stale extracts.
  Publisher note 8 explains why: the up-to-six-month carry-forward of missing practice data was discontinued
  from 2026/27, so a "latest data date" per practice no longer has meaning.

**Confirming the June notebook.** The hypothesis that June represents "the newer publication structure" holds.
What the earlier notebooks could not establish — because they only had two releases — is that the older
structure is stable rather than a March quirk. May 2025 settles that.

**Naming the eras.**

| era | releases seen | shape |
|---|---|---|
| **Era A — split files, rolling 13-month window** | May 2025, March 2026 | ~11 CSVs, category encoded inside `Measure`, 13 periods per file, measures reported at Sub-ICB/ICB/Region/Country |
| **Era B — consolidated files, single period** | June 2026 (2026/27 onwards) | 4 CSVs + mapping, explicit `BREAKDOWN` + dimension columns, 1 period per file, Sub-ICB measures at Sub-ICB only |

---
## 4. Row grain and candidate keys

**Question.** What does one row represent in each file, and what is the smallest column set that identifies it?

**Test.** Start from every non-value column, confirm uniqueness, then greedily drop columns — descriptive first
(`NAME`, `INDICATOR`), then secondary identifiers (ONS codes where an ODS code exists), then highest cardinality
— keeping a drop only when uniqueness survives. The date column is pinned into the key even where it is
constant, because in June each file holds a single period and would otherwise produce a key that looks smaller
than the concept really is.

In [10]:
keys = []
for (rel, fam), df in RAW_FRAMES.items():
    date_col = io.date_column(df)
    # LATEST_DATA_SUBMISSION is an attribute of a practice, not a reporting period, so it is not pinned.
    pin = [date_col] if date_col and io.normalise_column_name(date_col) != "LATEST_DATA_SUBMISSION" else None
    keys.append({"release": rel, "family": fam, **prof.minimal_key(df, require=pin)})

key_table = pd.DataFrame(keys).sort_values(["family", "release"]).reset_index(drop=True)
key_table.to_csv(QA / "candidate_keys.csv", index=False)
key_table[["release", "family", "key_found", "n_key_columns", "key", "duplicate_rows"]]

,release,family,key_found,n_key_columns,key,duplicate_rows
0,2025-05,la_rate,True,4,ORG_TYPE + ONS_CODE + ACH_DATE + MEASURE,0
1,2026-03,la_rate,True,4,ORG_TYPE + ONS_CODE + ACH_DATE + MEASURE,0
2,2026-06,la_rate,True,4,ORG_TYPE + ONS_CODE + ACH_DATE + MEASURE,0
3,2025-05,nhs_rate,True,3,ORG_CODE + ACH_DATE + MEASURE,0
4,2026-03,nhs_rate,True,3,ORG_CODE + ACH_DATE + MEASURE,0
5,2026-06,nhs_rate,True,3,ORG_CODE + ACH_DATE + MEASURE,0
6,2025-05,practice_data_date,True,1,PRACTICE_CODE,0
7,2026-03,practice_data_date,True,1,PRACTICE_CODE,0
8,2025-05,practice_mapping,True,2,EXTRACT_DATE + PRACTICE_CODE,0
9,2026-03,practice_mapping,True,2,EXTRACT_DATE + PRACTICE_CODE,0


In [11]:
# Is every column of the June Sub-ICB key genuinely load-bearing?
sub_icb = RAW_FRAMES[("2026-06", "sub_icb_consolidated")]
sub_icb_key = ["ACH_DATE", "ODS_CODE", "MEASURE", "BREAKDOWN",
               "AGE", "GENDER", "ETHNICITY", "DEMENTIA_TYPE", "RESIDENTIAL_TYPE"]
prof.key_sensitivity(sub_icb, sub_icb_key)

,removed_column,duplicate_rows
0,(none),0
1,ACH_DATE,0
2,ODS_CODE,11550
3,MEASURE,2650
4,BREAKDOWN,1166
5,AGE,6784
6,GENDER,3710
7,ETHNICITY,742
8,DEMENTIA_TYPE,742
9,RESIDENTIAL_TYPE,530


In [12]:
# And the June practice file: MEASURE was dropped - is BREAKDOWN really sufficient?
practice = RAW_FRAMES[("2026-06", "practice_measures")]
print(practice[["MEASURE", "BREAKDOWN"]].drop_duplicates().to_string(index=False))
print("\nBREAKDOWN -> MEASURE is a function:",
      bool((practice.groupby("BREAKDOWN")["MEASURE"].nunique() == 1).all()))

          MEASURE                 BREAKDOWN
DEMENTIA_REGISTER    DEMENTIA_REGISTER_0_64
DEMENTIA_REGISTER DEMENTIA_REGISTER_65_PLUS
         PAT_LIST             PAT_LIST_0_64
         PAT_LIST          PAT_LIST_65_PLUS
          REVIEWS   DIAG_DECLINED_CARE_PLAN
          REVIEWS   DIAG_RECEIVED_CARE_PLAN
          REVIEWS     DIAG_RECEIVED_MED_REV

BREAKDOWN -> MEASURE is a function: True


**Result — one sentence per family.**

| family | one row is… | key |
|---|---|---|
| `nhs_rate` | one diagnosis-rate measure for one NHS organisation at one period end | `ORG_CODE + ACH_DATE + MEASURE` |
| `la_rate` | one diagnosis-rate measure for one local-government geography **at one tier** | `ORG_TYPE + ONS_CODE + ACH_DATE + MEASURE` |
| Era-A Sub-ICB breakdown files | one category count for one Sub-ICB at one period end | `ACH_DATE + SUB_ICB_ODS_CODE + Measure` |
| Era-A multi-level measure files | one measure for one NHS organisation at any of four levels | `ACH_DATE + ORG_CODE + Measure` |
| `sub_icb_consolidated` (June) | one measure/breakdown/category count for one Sub-ICB | 9 columns — `ACH_DATE + ODS_CODE + MEASURE + BREAKDOWN + AGE + GENDER + ETHNICITY + DEMENTIA_TYPE + RESIDENTIAL_TYPE` |
| `practice_measures` (June) | one data item for one GP practice | `ACH_DATE + ODS_CODE + BREAKDOWN` |
| `practice_mapping` | one GP practice at the mapping extract date | `EXTRACT_DATE + PRACTICE_CODE` |
| `practice_data_date` | one GP practice (the date is an attribute, not part of the grain) | `PRACTICE_CODE` |

Three things worth pulling out:

* **`ORG_TYPE` is mandatory in the `la_rate` key and only there.** 132 ONS codes appear at both `LTLA` and
  `UTLA` in the same period (unitary authorities are their own upper tier). Joining `la_rate` on ONS code alone
  doubles those geographies. In `nhs_rate` the ODS code is unique across levels, so `ORG_TYPE` is descriptive.
* **The June Sub-ICB key needs eight columns plus the period.** The sensitivity table shows every removal
  reintroduces duplicates *except* `ACH_DATE`, which is redundant only because the June file holds a single
  period — it is still part of the concept and must stay in the key. `BREAKDOWN` is not redundant with the
  dimension columns: for `FRAILTY`, `PRESCRIBING` and `REFERRALS` the breakdown *is* the data item while all
  five dimension columns read `ALL`.
* **In the June practice file, `MEASURE` is a pure roll-up of `BREAKDOWN`** and carries no identifying
  information. This confirms the observation in the June notebook, and also confirms the dictionary is wrong:
  it documents `MEASURE = PAT_LIST_65_PLUS` and `MEASURE = REVIEW`, where the CSV uses `PAT_LIST` and `REVIEWS`.

**Implication.** Silver's natural grain is one observation per
`release × period × org_level × org_code × measure × breakdown × (age, gender, ethnicity, dementia_type, residential_type)`.
Era-A rows fill the dimension columns from the decomposed `Measure` string; Era-B rows fill them directly.

---
## 5. Temporal coverage and overlap

**Question.** Exactly what periods does each release contain, and how much do releases overlap?

**Why it matters.** This decides whether every historical release has to be ingested or only selected ones.

In [13]:
coverage = []
for (rel, fam), df in RAW_FRAMES.items():
    date_col = io.date_column(df)
    if date_col is None or io.normalise_column_name(date_col) == "LATEST_DATA_SUBMISSION":
        continue
    dates = io.parse_dates(df[date_col]).dropna()
    coverage.append({
        "release": rel, "family": fam, "date_column": date_col,
        "n_periods": dates.nunique(),
        "first_period": dates.min().date(), "last_period": dates.max().date(),
        "rows": len(df),
    })

coverage = pd.DataFrame(coverage).sort_values(["release", "family"]).reset_index(drop=True)
coverage.to_csv(QA / "temporal_coverage.csv", index=False)
coverage

,release,family,date_column,n_periods,first_period,last_period,rows
0,2025-05,la_rate,ACH_DATE,13,2024-05-31,2025-05-31,28470
1,2025-05,nhs_rate,ACH_DATE,13,2024-05-31,2025-05-31,10140
2,2025-05,practice_mapping,EXTRACT_DATE,1,2025-06-01,2025-06-01,6215
3,2025-05,sicbl_age_sex,ACH_DATE,13,2024-05-31,2025-05-31,24804
4,2025-05,sicbl_cog_imp,ACH_DATE,12,2024-06-30,2025-05-31,41184
5,2025-05,sicbl_comor_pall_care,ACH_DATE,13,2024-05-31,2025-05-31,6084
6,2025-05,sicbl_dem_type,ACH_DATE,13,2024-05-31,2025-05-31,8480
7,2025-05,sicbl_ethnicity,ACH_DATE,13,2024-05-31,2025-05-31,11024
8,2025-05,sicbl_incidence_onset_delirium,ACH_DATE,13,2024-05-31,2025-05-31,8424
9,2025-05,sicbl_res_type,ACH_DATE,13,2024-05-31,2025-05-31,8268


In [14]:
# Period sets per release, and the overlap matrix
period_sets = {}
for (rel, fam), df in RAW_FRAMES.items():
    col = io.date_column(df)
    if col is None or io.normalise_column_name(col) != "ACH_DATE":
        continue
    period_sets.setdefault(rel, set()).update(io.parse_dates(df[col]).dropna().unique())

overlap = pd.DataFrame(
    [[len(period_sets[a] & period_sets[b]) for b in releases] for a in releases],
    index=releases, columns=releases)
overlap.to_csv(QA / "release_overlap_matrix.csv")

for rel in releases:
    months = sorted(pd.Series(list(period_sets[rel])).dt.strftime("%Y-%m"))
    print(f"{rel}: {len(months):>2} periods  {months[0]} .. {months[-1]}")
print()
overlap

2025-05: 13 periods  2024-05 .. 2025-05
2026-03: 13 periods  2025-03 .. 2026-03
2026-06:  1 periods  2026-06 .. 2026-06



,2025-05,2026-03,2026-06
2025-05,13,3,0
2026-03,3,13,0
2026-06,0,0,1


In [15]:
# Is the union of the three releases continuous?
all_periods = sorted(set().union(*period_sets.values()))
span = pd.period_range(pd.Period(all_periods[0], "M"), pd.Period(all_periods[-1], "M"), freq="M")
held = {pd.Period(p, "M") for p in all_periods}
missing = [str(p) for p in span if p not in held]
print(f"Span {span[0]} .. {span[-1]}  ({len(span)} months)")
print(f"Held: {len(held)}   Missing: {missing}")

Span 2024-05 .. 2026-06  (26 months)
Held: 24   Missing: ['2026-04', '2026-05']


**Result.**

* **Era A releases each carry a rolling 13-month window** ending at the publication month:
  May 2025 covers 2024-05 → 2025-05, March 2026 covers 2025-03 → 2026-03. The suspicion in the March notebook
  that "most files contain a 13-month time series" is correct, and May 2025 shows it is the era's standing
  behaviour rather than a one-off.
* **One Era-A exception**: cognitive impairment in the May 2025 release has **12** periods, not 13 — it starts
  at 2024-06. The measure had not been published for 2024-05.
* **Era B carries a single period.** All four June 2026 CSVs contain only 2026-06-30.
* **The March 2026 and June 2026 releases have zero overlapping periods.** So do May 2025 and June 2026. The
  only overlap anywhere in the supplied data is **May 2025 × March 2026 = 3 months** (2025-03, 2025-04, 2025-05).
* **The union has a hole.** 2026-04 and 2026-05 are in no supplied release, because the March release's window
  stops at March and the June release only carries June.

**Implication — this is the single most consequential operational finding.** Under Era A a yearly download was
enough: any release regenerated the previous twelve months. Under Era B **every monthly release must be ingested
or that month is lost forever**, since nothing later republishes it. The ingestion schedule has to change at the
era boundary, and the two missing 2026 months have to be back-filled from the April and May 2026 publications.

---
## 6. Do overlapping releases revise historical values?

**Question.** Where two releases describe the same period, organisation, measure and breakdown, do they agree?

**Why it matters.** It decides the duplicate-resolution rule. If later releases revise history, silver needs
release-aware versioning and a "latest wins" resolution. If they never revise, archived releases are
interchangeable and ingestion can be far simpler.

**Test.** Project every file onto a shared QA observation shape (`src/pcdd_compare.py`), restrict to the
overlapping periods, and full-outer-join on
`family + org_level + org_code + ons_code + period + measure + breakdown + all five dimension columns`.
Compare the **raw token first** (so `*` vs `3` registers as a state change rather than a numeric difference)
and then numerically. Counting left-only and right-only rows also tests whether the two releases agree on which
observations *exist*, not just on their values.

In [16]:
observations = pd.concat(
    [cmp.to_observations(df, rel, fam) for (rel, fam), df in RAW_FRAMES.items()
     if fam in cmp.FAMILY_MAP],
    ignore_index=True,
)
observations.to_parquet(QA / "observations_all_releases.parquet", index=False)
print(f"{len(observations):,} observations projected")
observations.groupby(["release", "family"]).size().unstack(0).fillna(0).astype(int)

359,346 observations projected


release,2025-05,2026-03,2026-06
family,,,
la_rate,28470,29415,2295
nhs_rate,10140,10140,750
practice_measures,0,0,42616
sicbl_age_sex,24804,24804,0
sicbl_cog_imp,41184,44616,0
sicbl_comor_pall_care,6084,6084,0
sicbl_dem_type,8480,10812,0
sicbl_ethnicity,11024,11024,0
sicbl_incidence_onset_delirium,8424,9984,0


In [17]:
summary, matched = cmp.compare_observations(
    observations[observations["release"] == "2025-05"],
    observations[observations["release"] == "2026-03"],
)
summary.to_csv(QA / "revision_comparison_2025-05_vs_2026-03.csv", index=False)
summary

,family,matched,left_only,right_only,identical,revised,state_change,pct_identical_of_matched
0,la_rate,6570,0,0,6570,0,0,100.0
1,nhs_rate,2340,0,0,2340,0,0,100.0
2,sicbl_age_sex,5724,0,0,5724,0,0,100.0
3,sicbl_cog_imp,10296,0,0,10296,0,0,100.0
4,sicbl_comor_pall_care,1404,0,0,1404,0,0,100.0
5,sicbl_dem_type,2332,0,0,2332,0,0,100.0
6,sicbl_ethnicity,2544,0,0,2544,0,0,100.0
7,sicbl_incidence_onset_delirium,2184,0,0,2184,0,0,100.0
8,sicbl_res_type,1908,0,0,1908,0,0,100.0


In [18]:
totals = summary[["matched", "left_only", "right_only", "identical", "revised", "state_change"]].sum()
print(totals.to_string())
print(f"\nIdentical share of matched observations: {totals['identical'] / totals['matched']:.4%}")
print(f"Observations present in only one of the two releases: {totals['left_only'] + totals['right_only']}")

matched         35302
left_only           0
right_only          0
identical       35302
revised             0
state_change        0

Identical share of matched observations: 100.0000%
Observations present in only one of the two releases: 0


**Result — historical observations are not revised.**

Across **35,302 matched observations** spanning 3 overlapping months and all 9 Era-A analytical families:

* **100.0000% are byte-identical.** Zero revised, zero state changes.
* **Zero left-only and zero right-only rows.** The two releases agree exactly on which observations exist, not
  merely on their values — including the ragged months where a measure had not yet been introduced.

This is a strong result, but it needs an honest boundary drawn around it:

* It is proven **within Era A only**, over a **3-month** overlap, between releases **10 months apart**.
* It says nothing about Era B, because **no Era-B period overlaps any other supplied release**. With a
  single-period publication model there may never be an overlap to test, which means the no-revision property
  cannot be assumed for 2026/27 — it has to be re-tested the first time two Era-B releases share a period
  (for example if a month is ever re-issued after a correction).

**Implication.** For Era A, archived releases are interchangeable: ingest whichever covers a period and use the
first one that does. Keep `source_release` on every silver row anyway, so the claim stays auditable and so a
future revision is detectable rather than silently overwriting. A conservative "latest release wins" rule costs
nothing today and protects against an Era-B correction later.

---
## 7. Measures and dimensions — the semantic model

**Question.** What measures and categories exist, how are they encoded in each era, and which are genuinely the
same concept?

**Why it matters.** This is the crosswalk the silver layer will be built on. Labels that look alike are not
evidence of equivalence.

In [19]:
measure_inventory = (
    observations.groupby(["release", "family", "measure", "breakdown"])
    .agg(rows=("value_raw", "size"),
         periods=("period_end", "nunique"),
         orgs=("org_code", "nunique"))
    .reset_index()
)
measure_inventory.to_csv(QA / "measure_inventory.csv", index=False)

print("Distinct measure/breakdown pairs per release:")
print(measure_inventory.groupby("release")[["measure"]].size().to_frame("measure_breakdown_pairs"))
measure_inventory.groupby(["release", "family"]).size().unstack(0).fillna(0).astype(int)

Distinct measure/breakdown pairs per release:
         measure_breakdown_pairs
release                         
2025-05                       80
2026-03                       80
2026-06                       41


release,2025-05,2026-03,2026-06
family,,,
la_rate,5,5,5
nhs_rate,5,5,5
practice_measures,0,0,7
sicbl_age_sex,18,18,0
sicbl_cog_imp,22,22,0
sicbl_comor_pall_care,3,3,0
sicbl_dem_type,8,8,0
sicbl_ethnicity,8,8,0
sicbl_incidence_onset_delirium,5,5,0


In [20]:
# When did each Era-A measure first appear? Pool both Era-A releases and count distinct
# organisations per period, so the 3 overlapping months are not double-counted.
era_a = observations[observations["release"].isin(["2025-05", "2026-03"])]
timeline = (era_a.assign(period=era_a["period_end"].dt.strftime("%Y-%m"))
            .pivot_table(index=["family", "measure"], columns="period",
                         values="org_code", aggfunc="nunique", fill_value=0))
timeline.to_csv(QA / "measure_availability_timeline.csv")

all_periods_a = list(timeline.columns)
gaps = []
for (family, measure), row in timeline.iterrows():
    present = [p for p in all_periods_a if row[p] > 0]
    if len(present) < len(all_periods_a):
        gaps.append({"family": family, "measure": measure,
                     "first_period_with_data": present[0] if present else "-",
                     "missing_periods": len(all_periods_a) - len(present)})
gaps = pd.DataFrame(gaps)
print(f"{len(timeline)} Era-A measures; {len(gaps)} do not cover all {len(all_periods_a)} periods "
      f"({all_periods_a[0]} .. {all_periods_a[-1]})")
(gaps.groupby(["first_period_with_data", "family"])
     .agg(measures=("measure", "size"), example=("measure", "first"))
     .reset_index())

80 Era-A measures; 27 do not cover all 23 periods (2024-05 .. 2026-03)


,first_period_with_data,family,measures,example
0,2024-06,sicbl_cog_imp,22,MCI_FEMALE_AGED_40_44
1,2024-06,sicbl_dem_type,2,FRONTOTEMPORAL
2,2025-04,sicbl_dem_type,2,OTHER_SPECIFIED_DEMENTIA_TYPES
3,2025-04,sicbl_incidence_onset_delirium,1,DELIRIUM_12M


In [21]:
# Category vocabularies: Era A encodes them in `Measure`, Era B in dimension columns
era_b = RAW_FRAMES[("2026-06", "sub_icb_consolidated")]
vocab = {
    "ethnicity": (set(RAW_FRAMES[("2026-03", "sicbl_ethnicity")]["Measure"]),
                  set(era_b["ETHNICITY"]) - {"ALL"}),
    "dementia_type": (set(RAW_FRAMES[("2026-03", "sicbl_dem_type")]["Measure"]),
                      set(era_b["DEMENTIA_TYPE"]) - {"ALL"}),
    "residential_type": (set(RAW_FRAMES[("2026-03", "sicbl_res_type")]["Measure"]),
                         set(era_b["RESIDENTIAL_TYPE"]) - {"ALL"}),
}
pd.DataFrame([
    {"dimension": k, "era_A_categories": len(a), "era_B_categories": len(b),
     "identical_vocabulary": a == b,
     "only_era_A": ", ".join(sorted(a - b)) or "-", "only_era_B": ", ".join(sorted(b - a)) or "-"}
    for k, (a, b) in vocab.items()
])

,dimension,era_A_categories,era_B_categories,identical_vocabulary,only_era_A,only_era_B
0,ethnicity,8,8,True,-,-
1,dementia_type,8,8,True,-,-
2,residential_type,6,6,True,-,-


In [22]:
# Age/sex is the dimension that genuinely changed shape
era_a_age = sorted(RAW_FRAMES[("2026-03", "sicbl_age_sex")]["Measure"].unique())
print("Era A age/sex measures (dementia register):")
print(" ", ", ".join(era_a_age))
print("\nEra B AGE x GENDER for DEMENTIA_REGISTER:")
reg = era_b[(era_b["MEASURE"] == "DEMENTIA_REGISTER") & (era_b["BREAKDOWN"] == "AGE_GENDER")]
print("  AGE   :", ", ".join(sorted(reg["AGE"].unique(), key=lambda s: (len(s), s))))
print("  GENDER:", ", ".join(sorted(reg["GENDER"].unique())))
print("\nGENDER casing by measure across the whole June Sub-ICB file:")
print(era_b[era_b["BREAKDOWN"] == "AGE_GENDER"]
      .groupby("MEASURE")["GENDER"].apply(lambda s: ", ".join(sorted(s.unique()))).to_string())

Era A age/sex measures (dementia register):
  ALL_AGED_65_69, ALL_AGED_70_74, ALL_AGED_75_79, ALL_AGED_80_84, ALL_AGED_85_89, ALL_AGED_90_PLUS, FEMALE_AGED_65_69, FEMALE_AGED_70_74, FEMALE_AGED_75_79, FEMALE_AGED_80_84, FEMALE_AGED_85_89, FEMALE_AGED_90_PLUS, MALE_AGED_65_69, MALE_AGED_70_74, MALE_AGED_75_79, MALE_AGED_80_84, MALE_AGED_85_89, MALE_AGED_90_PLUS

Era B AGE x GENDER for DEMENTIA_REGISTER:
  AGE   : 0_39, 40_44, 45_49, 50_54, 55_59, 60_64, 65_69, 70_74, 75_79, 80_84, 85_89, 90_PLUS
  GENDER: Female, Male

GENDER casing by measure across the whole June Sub-ICB file:
MEASURE
DEMENTIA_REGISTER    Female, Male
MCI                  Female, Male
PAT_LIST             FEMALE, MALE


### 7.1 Evidence for the age/sex crosswalk

Labels alone would not settle whether Era A's `FEMALE_AGED_65_69` is the same thing as Era B's
`AGE=65_69, GENDER=Female`. Two arithmetic identities can be tested instead.

In [23]:
def to_num(s):
    return pd.to_numeric(s, errors="coerce")

# (a) Era A: is ALL_AGED_<band> exactly FEMALE + MALE?
a = RAW_FRAMES[("2026-03", "sicbl_age_sex")].copy()
a["v"] = to_num(a["Value"])
a["sex"] = a["Measure"].str.extract(r"^(ALL|FEMALE|MALE)_AGED_")
a["band"] = a["Measure"].str.replace(r"^(ALL|FEMALE|MALE)_AGED_", "", regex=True)
piv = a.pivot_table(index=["ACH_DATE", "SUB_ICB_ODS_CODE", "band"], columns="sex", values="v")
exact = piv["ALL"] == piv["FEMALE"] + piv["MALE"]
print(f"(a) Era A, all 13 periods: ALL == FEMALE + MALE for {exact.sum():,}/{len(piv):,} cells ({exact.mean():.4%})")

# (b) Does the Sub-ICB 65+ register sum reconcile to nhs_rate DEMENTIA_REGISTER_65_PLUS, in both eras?
bands_65 = ["65_69", "70_74", "75_79", "80_84", "85_89", "90_PLUS"]

mar_sum = (a[(a["ACH_DATE"] == "31-Mar-26") & a["Measure"].str.startswith("ALL_AGED_")]
           .groupby("SUB_ICB_ODS_CODE")["v"].sum())
nhs_mar = RAW_FRAMES[("2026-03", "nhs_rate")].copy(); nhs_mar["v"] = to_num(nhs_mar["VALUE"])
nhs_mar = nhs_mar.query("ACH_DATE == '31-Mar-26' and ORG_TYPE == 'SUB_ICB_LOC' "
                        "and MEASURE == 'DEMENTIA_REGISTER_65_PLUS'").set_index("ORG_CODE")["v"]

b = era_b.copy(); b["v"] = to_num(b["VALUE"])
jun_sum = (b[(b["MEASURE"] == "DEMENTIA_REGISTER") & (b["BREAKDOWN"] == "AGE_GENDER")
             & b["AGE"].isin(bands_65)].groupby("ODS_CODE")["v"].sum())
nhs_jun = RAW_FRAMES[("2026-06", "nhs_rate")].copy(); nhs_jun["v"] = to_num(nhs_jun["VALUE"])
nhs_jun = nhs_jun.query("ORG_TYPE == 'SUB_ICB_LOC' and MEASURE == 'DEMENTIA_REGISTER_65_PLUS'"
                        ).set_index("ORG_CODE")["v"]

for label, breakdown_sum, headline in [("Era A (Mar 2026)", mar_sum, nhs_mar),
                                       ("Era B (Jun 2026)", jun_sum, nhs_jun)]:
    joined = pd.concat([breakdown_sum.rename("sum"), headline.rename("headline")], axis=1).dropna()
    match = (joined["sum"] == joined["headline"])
    print(f"(b) {label}: Sub-ICB 65+ breakdown sum == nhs_rate register for "
          f"{match.sum()}/{len(joined)} Sub-ICBs, max |diff| {float((joined['sum'] - joined['headline']).abs().max()):.0f}")

(a) Era A, all 13 periods: ALL == FEMALE + MALE for 8,268/8,268 cells (100.0000%)
(b) Era A (Mar 2026): Sub-ICB 65+ breakdown sum == nhs_rate register for 106/106 Sub-ICBs, max |diff| 0
(b) Era B (Jun 2026): Sub-ICB 65+ breakdown sum == nhs_rate register for 106/106 Sub-ICBs, max |diff| 0


In [24]:
# (c) Publisher note 2: breakdown totals "may not be equal". How unequal, in each era?
jun_total = (b[(b["MEASURE"] == "DEMENTIA_REGISTER") & (b["BREAKDOWN"] == "AGE_GENDER")]
             .groupby("ODS_CODE")["v"].sum())
out = []
for breakdown in ["ETHNICITY", "DEMENTIA_TYPE", "RESIDENCE_TYPE"]:
    t = b[(b["MEASURE"] == "DEMENTIA_REGISTER") & (b["BREAKDOWN"] == breakdown)].groupby("ODS_CODE")["v"].sum()
    d = t - jun_total
    out.append({"era": "B (Jun 2026)", "breakdown": breakdown, "sub_icbs_equal": int((d == 0).sum()),
                "n": len(d), "max_abs_diff": int(d.abs().max()), "suppressed_cells": 0})

inc = RAW_FRAMES[("2026-03", "sicbl_incidence_onset_delirium")].copy(); inc["v"] = to_num(inc["Value"])
reg_all = inc.query("ACH_DATE == '31-Mar-26' and ORG_TYPE == 'SUB_ICB' and Measure == 'DEMENTIA_REGISTER'"
                    ).set_index("ORG_CODE")["v"]
for fam, breakdown in [("sicbl_ethnicity", "ETHNICITY"), ("sicbl_dem_type", "DEMENTIA_TYPE"),
                       ("sicbl_res_type", "RESIDENCE_TYPE")]:
    d0 = RAW_FRAMES[("2026-03", fam)].query("ACH_DATE == '31-Mar-26'").copy()
    d0["v"] = to_num(d0["Value"])
    t = d0.groupby("SUB_ICB_ODS_CODE")["v"].sum()
    d = t - reg_all
    out.append({"era": "A (Mar 2026)", "breakdown": breakdown, "sub_icbs_equal": int((d == 0).sum()),
                "n": len(d), "max_abs_diff": int(d.abs().max()),
                "suppressed_cells": int((d0["Value"] == "*").sum())})
pd.DataFrame(out)

,era,breakdown,sub_icbs_equal,n,max_abs_diff,suppressed_cells
0,B (Jun 2026),ETHNICITY,99,106,2,0
1,B (Jun 2026),DEMENTIA_TYPE,99,106,2,0
2,B (Jun 2026),RESIDENCE_TYPE,99,106,2,0
3,A (Mar 2026),ETHNICITY,98,106,2,0
4,A (Mar 2026),DEMENTIA_TYPE,15,106,11,4
5,A (Mar 2026),RESIDENCE_TYPE,15,106,9,87


**Result.**

* **(a) `ALL_AGED_<band>` is exactly `FEMALE + MALE` in 100.0000% of 8,268 Era-A cells.** The Era-A `ALL_*`
  rows are pure derivations, so Era B dropping the all-sex row loses nothing — it is recomputable.
* **(b) The Sub-ICB age/sex breakdown reconciles to the `nhs_rate` 65+ register exactly, in both eras**
  (106/106 Sub-ICBs, max difference 0). That is the strongest available evidence that the age/gender
  dementia-register concept survived the era change unchanged, and it also confirms publisher note 2: the
  headline register total *is* the sum of the age/sex breakdown.
* **(c) The other three breakdowns do not sum to the same total** — exactly as note 2 warns. In Era B they
  differ from the age/gender total by at most 2 patients in 7 of 106 Sub-ICBs. In Era A the comparison has to be
  made against the *all-age* register, not the 65+ one, because **Era A's ethnicity / dementia-type /
  residential-type files cover the whole dementia register while its age/sex file covers 65+ only.** Era B's
  `AGE_GENDER` breakdown covers all ages (0_39 upward), so in Era B all four breakdowns finally share one
  denominator.

### 7.2 Cross-era crosswalk, with stated confidence

In [25]:
crosswalk = pd.DataFrame([
    # concept, era A encoding, era B encoding, confidence, basis
    ("Diagnosis rate measures (5)", "pcdem-nhs-rate / pcdem-la-rate MEASURE",
     "identical", "confirmed", "Identical schema, identical measure names, identical dictionary definitions."),
    ("Dementia register by age & sex, 65+", "sicbl_age_sex: FEMALE|MALE_AGED_<band>",
     "sub_icb: DEMENTIA_REGISTER / AGE_GENDER / AGE=<band>, GENDER=Female|Male",
     "confirmed", "Both eras' 65+ sums reconcile exactly to nhs_rate DEMENTIA_REGISTER_65_PLUS (106/106)."),
    ("Dementia register by age & sex, all-sex total", "sicbl_age_sex: ALL_AGED_<band>",
     "derive as Female + Male", "confirmed", "ALL == FEMALE + MALE in 100.0000% of Era-A cells."),
    ("Dementia register under 65 by age & sex", "not published",
     "sub_icb: AGE in 0_39..60_64", "new in Era B", "No Era-A equivalent exists."),
    ("Dementia register by ethnicity", "sicbl_ethnicity: Measure = <category>",
     "sub_icb: DEMENTIA_REGISTER / ETHNICITY / ETHNICITY=<category>",
     "confirmed (labels) / probable (denominator)",
     "Category vocabulary byte-identical; both are all-age register. Era A has no suppression here."),
    ("Dementia register by dementia type", "sicbl_dem_type: Measure = <category>",
     "sub_icb: DEMENTIA_REGISTER / DEMENTIA_TYPE / DEMENTIA_TYPE=<category>",
     "confirmed (labels) / probable (values)",
     "Vocabulary byte-identical; ~1% of Era-A cells suppressed so totals are not reconstructable."),
    ("Dementia register by residential type", "sicbl_res_type: Measure = <category>",
     "sub_icb: DEMENTIA_REGISTER / RESIDENCE_TYPE / RESIDENTIAL_TYPE=<category>",
     "labels confirmed / values NOT comparable",
     "Vocabulary identical, but ~14% of Era-A cells suppressed vs 0% in Era B; Era-B publishes values 0-4."),
    ("Mild cognitive impairment by age & sex", "sicbl_cog_imp: MCI_FEMALE|MALE_AGED_<band>",
     "sub_icb: MCI / AGE_GENDER", "confirmed (concept) / probable (values)",
     "Same 11 bands x 2 sexes; ~1% Era-A suppression. Era A also published ICB/Region/Country levels."),
    ("Patient list by age & sex", "sicbl_incidence...: PAT_LIST_ALL (single all-ages figure)",
     "sub_icb: PAT_LIST / AGE_GENDER (12 bands x 2 sexes)", "probable",
     "Era B is a finer breakdown of the same population; the all-ages total should be the sum but cannot be "
     "verified without an overlapping period."),
    ("Incidence", "sicbl_incidence...: INCIDENCE (1 calendar month)",
     "sub_icb: INCIDENCE (1 quarter)", "NOT comparable",
     "Publisher note 7: reporting period changed from 1 month to 3 months from 2026/27."),
    ("Young onset", "sicbl_incidence...: YOUNG_ONSET", "sub_icb: YOUNG_ONSET", "confirmed",
     "Identical dictionary definition in both eras."),
    ("Delirium in last 12 months", "sicbl_incidence...: DELIRIUM_12M", "sub_icb: DELIRIUM_12M", "confirmed",
     "Identical definition. Era A only from 2025-04 onwards."),
    ("Palliative care", "sicbl_comor_pall_care: PALLIATIVE_CARE", "sub_icb: PALLIATIVE_CARE", "confirmed",
     "Identical definition (65+ on QOF palliative care register)."),
    ("Comorbidities", "sicbl_comor_pall_care: COMORBIDITIES", "not published", "discontinued",
     "No Era-B equivalent."),
    ("Antipsychotic prescribing", "pcdem-prac-anti-psy (practice level; not in local source set)",
     "sub_icb: PRESCRIBING / PSY_DIAG*, NO_PSY_DIAG*", "uncertain",
     "Same breakdown names, but Era A published them per practice and Era B per Sub-ICB."),
    ("Dementia drug prescribing", "not published",
     "sub_icb: PRESCRIBING / DON_RIV_GAL_12M, MEM_12M, MEM_PLUS_DON_RIV_GAL_12M", "new in Era B", ""),
    ("Frailty & falls", "not published", "sub_icb: FRAILTY / 5 breakdowns", "new in Era B", ""),
    ("Memory clinic referrals", "not published (earlier referral measures differ)",
     "sub_icb: REFERRALS / 2 breakdowns", "new in Era B",
     "Publisher note 6: introduced in 2026/27 and explicitly not comparable to previous referral measures."),
    ("Care plans & medication reviews", "pcdem-prac-ass-plans (not in local source set)",
     "practice: REVIEWS / 3 breakdowns", "uncertain",
     "Era-A file not supplied locally; note 4 warns these are cumulative Apr-Mar and drop between Q4 and Q1."),
], columns=["concept", "era_A_encoding", "era_B_encoding", "confidence", "basis"])

crosswalk.to_csv(QA / "measure_crosswalk.csv", index=False)
crosswalk.groupby("confidence").size().to_frame("concepts")

,concepts
confidence,
NOT comparable,1
confirmed,6
confirmed (concept) / probable (values),1
confirmed (labels) / probable (denominator),1
confirmed (labels) / probable (values),1
discontinued,1
labels confirmed / values NOT comparable,1
new in Era B,4
probable,1


The full crosswalk is written to `outputs/qa/measure_crosswalk.csv`. The confidence labels mean:

* **confirmed** — the equivalence is either definitional (identical dictionary wording) or demonstrated
  arithmetically here.
* **probable** — the concept is clearly the same but values cannot be proved equal, usually because suppression
  or the absence of an overlapping period makes the check impossible.
* **uncertain** — the Era-A source is not in the local set, or the organisational level changed.
* **NOT comparable / new / discontinued** — do not join these across the era boundary at all.

---
## 8. Geography and organisations

**Question.** How are organisations identified, what joins are possible, and what breaks historical comparison?

In [26]:
# Distinct entities are counted two ways: pooled over every period in the release, and at the
# release's own latest period. Where the two differ, the entity set changed inside the release.
org_levels = []
for (rel, fam), df in RAW_FRAMES.items():
    if "ORG_TYPE" not in df.columns:
        continue
    code_col = next((c for c in ("ORG_CODE", "ODS_CODE", "ONS_CODE") if c in df.columns), None)
    date_col = io.date_column(df)
    latest = df
    if date_col and io.normalise_column_name(date_col) == "ACH_DATE":
        periods = io.parse_dates(df[date_col])
        latest = df[periods == periods.max()]
    for level, n in df["ORG_TYPE"].value_counts().items():
        org_levels.append({
            "release": rel, "family": fam, "org_type": level, "rows": int(n),
            "entities_pooled": int(df.loc[df["ORG_TYPE"] == level, code_col].nunique()),
            "entities_latest_period": int(latest.loc[latest["ORG_TYPE"] == level, code_col].nunique()),
        })
org_levels = pd.DataFrame(org_levels)
org_levels.to_csv(QA / "organisation_levels.csv", index=False)

pd.concat([
    org_levels.pivot_table(index="org_type", columns="release", values="entities_pooled", aggfunc="max"),
    org_levels.pivot_table(index="org_type", columns="release", values="entities_latest_period", aggfunc="max"),
], axis=1, keys=["pooled over all periods", "at latest period"]).fillna("-")

pooled over all periods                 at latest period                
release                                2025-05 2026-03 2026-06          2025-05 2026-03 2026-06
org_type                                                                                       
COUNTRY                                    1.0     1.0       -              1.0     1.0       -
COUNTRY_GEOGRAPHICAL                       1.0     1.0     1.0              1.0     1.0     1.0
COUNTRY_RESPONSIBILITY                     1.0     1.0     1.0              1.0     1.0     1.0
GOR                                        9.0     9.0     9.0              9.0     9.0     9.0
ICB                                       42.0    42.0    36.0             42.0    42.0    36.0
LTLA                                     296.0   298.0   296.0            296.0   296.0   296.0
NHS_REGION                                 7.0     7.0     7.0              7.0     7.0     7.0
PRACTICE                                     -       -  6088.0                -       -  6088.0
REGION                                     7.0     7.0       -              7.0     7.0       -
SUB_ICB                                  106.0   106.0   106.0            106.0   106.0   106.0
SUB_ICB_LOC                              106.0   106.0   106.0            106.0   106.0   106.0
UTLA                                     132.0   155.0   153.0            132.0   153.0   153.0

In [27]:
# The three vocabularies used for the same four levels
vocabs = (org_levels.groupby(["family"])["org_type"]
          .apply(lambda s: " / ".join(sorted(s.unique()))).drop_duplicates())
vocabs.to_frame("ORG_TYPE vocabulary")

,ORG_TYPE vocabulary
family,
la_rate,COUNTRY_GEOGRAPHICAL / GOR / LTLA / UTLA
nhs_rate,COUNTRY_RESPONSIBILITY / ICB / NHS_REGION / SUB_ICB_LOC
practice_measures,PRACTICE
sicbl_cog_imp,COUNTRY / ICB / REGION / SUB_ICB
sub_icb_consolidated,SUB_ICB


In [28]:
# ICB reorganisation between March and June 2026
mar = RAW_FRAMES[("2026-03", "nhs_rate")]
mar = mar[mar["ACH_DATE"] == "31-Mar-26"]
jun = RAW_FRAMES[("2026-06", "nhs_rate")]
mar_icb = set(mar.loc[mar["ORG_TYPE"] == "ICB", "ORG_CODE"])
jun_icb = set(jun.loc[jun["ORG_TYPE"] == "ICB", "ORG_CODE"])
print(f"ICBs: March {len(mar_icb)} -> June {len(jun_icb)}")
print("  retired :", ", ".join(sorted(mar_icb - jun_icb)))
print("  new     :", ", ".join(sorted(jun_icb - mar_icb)))

mar_sub = set(mar.loc[mar["ORG_TYPE"] == "SUB_ICB_LOC", "ORG_CODE"])
jun_sub = set(jun.loc[jun["ORG_TYPE"] == "SUB_ICB_LOC", "ORG_CODE"])
print(f"\nSub-ICBs: March {len(mar_sub)} -> June {len(jun_sub)}; identical set: {mar_sub == jun_sub}")
print("  retired :", ", ".join(sorted(mar_sub - jun_sub)) or "-")
print("  new     :", ", ".join(sorted(jun_sub - mar_sub)) or "-")

ICBs: March 42 -> June 36
  retired : QH8, QHG, QJG, QM7, QMJ, QMM, QNQ, QNX, QRV, QU9, QUE, QXU
  new     : D7T5G, S0E4D, S1Y5D, S9B9J, T6Y0W, Z9B2Z

Sub-ICBs: March 106 -> June 106; identical set: False
  retired : D4U1Y
  new     : U2G6B


In [29]:
# How many Sub-ICBs were reassigned to a different ICB?
mar_map = RAW_FRAMES[("2026-03", "practice_mapping")][["SUB_ICB_LOCATION_CODE", "ICB_CODE"]].drop_duplicates()
jun_map = RAW_FRAMES[("2026-06", "practice_mapping")][["SUB_ICB_LOCATION_CODE", "ICB_CODE"]].drop_duplicates()
hierarchy = mar_map.merge(jun_map, on="SUB_ICB_LOCATION_CODE", how="outer",
                          suffixes=("_mar", "_jun"))
changed = hierarchy[(hierarchy["ICB_CODE_mar"] != hierarchy["ICB_CODE_jun"])
                    & hierarchy["ICB_CODE_mar"].notna() & hierarchy["ICB_CODE_jun"].notna()]
print(f"Sub-ICB -> ICB assignments that changed: {len(changed)} of {len(hierarchy)}")
changed.sort_values("ICB_CODE_jun").head(30)

Sub-ICB -> ICB assignments that changed: 23 of 108


,SUB_ICB_LOCATION_CODE,ICB_CODE_mar,ICB_CODE_jun
52,06Q,QH8,D7T5G
53,06T,QJG,D7T5G
54,07G,QH8,D7T5G
55,07H,QM7,D7T5G
94,99E,QH8,D7T5G
95,99F,QH8,D7T5G
96,99G,QH8,D7T5G
58,10Q,QU9,S0E4D
67,14Y,QU9,S0E4D
68,15A,QU9,S0E4D


In [30]:
# Local-authority geography: the count changes *inside* the March 2026 release
la = RAW_FRAMES[("2026-03", "la_rate")].copy()
la["period"] = io.parse_dates(la["ACH_DATE"]).dt.strftime("%Y-%m")
by_period = la.pivot_table(index="period", columns="ORG_TYPE", values="ONS_CODE", aggfunc="nunique")
la5 = RAW_FRAMES[("2025-05", "la_rate")].copy()
la5["period"] = io.parse_dates(la5["ACH_DATE"]).dt.strftime("%Y-%m")
pd.concat([la5.pivot_table(index="period", columns="ORG_TYPE", values="ONS_CODE", aggfunc="nunique"),
           by_period]).drop_duplicates().sort_index()

ORG_TYPE,COUNTRY_GEOGRAPHICAL,GOR,LTLA,UTLA
period,,,,
2024-05,1,9,296,132
2025-07,1,9,296,153


In [31]:
# Local-authority ONS codes are not stable either - does membership change, not just count?
ltla = la[la["ORG_TYPE"] == "LTLA"]
sets = {p: set(g["ONS_CODE"]) for p, g in ltla.groupby("period")}
baseline = sets[min(sets)]
for period in sorted(sets):
    added, removed = sets[period] - baseline, baseline - sets[period]
    if added or removed:
        print(f"{period}: +{sorted(added)}  -{sorted(removed)}")
        break
print("LTLA count is constant at", len(baseline), "in every period, but membership is not.")

2025-08: +['E08000038', 'E08000039']  -['E08000016', 'E08000019']
LTLA count is constant at 296 in every period, but membership is not.


In [32]:
# Identifier hygiene: England, and ONS codes shared across tiers
for rel in releases:
    for fam in ("nhs_rate", "la_rate"):
        df = RAW_FRAMES[(rel, fam)]
        country = df[df["ORG_TYPE"].str.contains("COUNTRY")]
        cols = [c for c in ("ORG_TYPE", "ORG_CODE", "ONS_CODE", "NAME") if c in df.columns]
        print(rel, fam, country[cols].drop_duplicates().to_dict("records"))

la_latest = RAW_FRAMES[("2026-06", "la_rate")]
shared = la_latest.groupby("ONS_CODE")["ORG_TYPE"].nunique()
print(f"\nONS codes appearing at more than one LA tier: {int((shared > 1).sum())} of {len(shared)}")

2025-05 nhs_rate [{'ORG_TYPE': 'COUNTRY_RESPONSIBILITY', 'ORG_CODE': 'ENG', 'ONS_CODE': 'ENG', 'NAME': 'ENGLAND'}]
2025-05 la_rate [{'ORG_TYPE': 'COUNTRY_GEOGRAPHICAL', 'ONS_CODE': 'E92000001', 'NAME': 'ENGLAND'}]
2026-03 nhs_rate [{'ORG_TYPE': 'COUNTRY_RESPONSIBILITY', 'ORG_CODE': 'ENG', 'ONS_CODE': 'ENG', 'NAME': 'ENGLAND'}]
2026-03 la_rate [{'ORG_TYPE': 'COUNTRY_GEOGRAPHICAL', 'ONS_CODE': 'E92000001', 'NAME': 'ENGLAND'}]
2026-06 nhs_rate [{'ORG_TYPE': 'COUNTRY_RESPONSIBILITY', 'ORG_CODE': 'ENG', 'ONS_CODE': 'ENG', 'NAME': 'ENGLAND'}]
2026-06 la_rate [{'ORG_TYPE': 'COUNTRY_GEOGRAPHICAL', 'ONS_CODE': 'E92000001', 'NAME': 'ENGLAND'}]

ONS codes appearing at more than one LA tier: 132 of 327


In [33]:
# Practice coverage chain, and where practices drop out
chain = []
for rel, measures_fam in [("2025-05", "practice_data_date"), ("2026-03", "practice_data_date"),
                          ("2026-06", "practice_measures")]:
    mapping = RAW_FRAMES[(rel, "practice_mapping")]
    other = RAW_FRAMES[(rel, measures_fam)]
    code_col = "PRACTICE_CODE" if "PRACTICE_CODE" in other.columns else "ODS_CODE"
    m, o = set(mapping["PRACTICE_CODE"]), set(other[code_col])
    chain.append({"release": rel, "compared_with": measures_fam,
                  "mapping_extract": mapping["EXTRACT_DATE"].iloc[0],
                  "mapping_practices": len(m), "other_practices": len(o),
                  "mapping_only": len(m - o), "unmappable": ", ".join(sorted(o - m)) or "-"})
pd.DataFrame(chain)

,release,compared_with,mapping_extract,mapping_practices,other_practices,mapping_only,unmappable
0,2025-05,practice_data_date,01Jun2025,6215,6148,68,C81028
1,2026-03,practice_data_date,01Mar2026,6169,6124,45,-
2,2026-06,practice_measures,01-Jul-26,6182,6088,94,-


**Result.**

* **Three different `ORG_TYPE` vocabularies for the same four NHS levels.** `nhs_rate` uses
  `SUB_ICB_LOC / ICB / NHS_REGION / COUNTRY_RESPONSIBILITY`; the Era-A multi-level measure files use
  `SUB_ICB / ICB / REGION / COUNTRY`; `la_rate` uses `LTLA / UTLA / GOR / COUNTRY_GEOGRAPHICAL`. These need a
  controlled vocabulary in silver.
* **England is identified inconsistently.** In `nhs_rate` its `ONS_CODE` is the string `ENG` (not an ONS code
  at all); in `la_rate` and the Era-A files it is `E92000001`. Any ONS-code join across the rate files loses or
  mismatches England.
* **An ICB reorganisation lands exactly on the era boundary.** 42 ICBs in March become 36 in June: twelve
  three-character ODS codes are retired and six new five-character codes appear. **23 Sub-ICBs are reassigned
  to a different ICB**, and the Sub-ICB set itself changes by one code (`D4U1Y` out, `U2G6B` in) while staying
  at 106. Sub-ICB codes are therefore *not* a stable spine either, though they are far more stable
  than ICB codes.
* **Any ICB-level time series that spans 2026-06 is a comparison of different organisations.** Region (7) and
  Sub-ICB (106) are the only NHS levels stable enough to trend across the boundary, and Sub-ICB only with the
  one-code caveat.
* **Local-authority coverage changes mid-series, inside a single release.** The pooled counts above are larger
  than the latest-period counts precisely because of this. UTLAs jump from 132 to 153 at **2025-07** — 21 county councils (`E10…`) and 2 metropolitan/unitary authorities (`E08…`) are added. This is
  visible in the March 2026 release and confirmed by May 2025, which shows 132 throughout. An England total
  built by summing UTLAs would therefore step up in July 2025 for purely definitional reasons.
* **LTLA ONS codes are re-issued mid-series.** The LTLA count never moves off 296, but at **2025-08**
  `E08000016` and `E08000019` are replaced by `E08000038` and `E08000039` — the same two authorities under new
  ONS codes. A count-based check would have missed this entirely; any LTLA time series joined on ONS code
  breaks at that point for those two authorities.
* **132 ONS codes appear at both `LTLA` and `UTLA`** in the same period. Unitary authorities are their own upper
  tier, so `ORG_TYPE` is part of the identity, and summing `la_rate` without filtering on tier double-counts.
* **Practice coverage is a funnel, and the mapping snapshot is always the widest part *by count*.** Comparing
  the mapping file with the release's other practice file: 6,215 vs 6,148 (May 2025, latest-submission file);
  6,169 vs 6,124 (March 2026, latest-submission file); 6,182 vs 6,088 (June 2026, practice measures file).
* **But the mapping is not a strict superset.** Its extract date is *after* the reporting period — 01 Jun 2025
  for a May 2025 release, 01 Jul 2026 for a June 2026 release — so a practice that closed in between can be
  measured and yet unmappable. May 2025 contains exactly one such practice (`C81028`, last submission
  2025-04-30). A pipeline that inner-joins measures to the mapping will silently drop practices like this. Publisher notes 9, 10 and 11 explain
  the shortfall — practices with no list size, practices that cannot be mapped to a Sub-ICB, and practices
  merged for disclosure control are removed from the measure files but stay in the mapping snapshot.
* **One June mapping row carries the literal string `NULL`** in all three hierarchy columns. Read with pandas'
  defaults this becomes `NaN` and looks like a missing value; it is actually a published sentinel. (The June
  notebook's "1 practice row with missing mapping fields" is right about the count and worth restating as a
  literal token.)
* **PCN disappears in Era B.** The June mapping file has no `PCN_CODE` / `PCN_NAME`, so practice → PCN is not
  available for 2026/27 from this publication.
* **Mapping-file name casing flipped** — Sub-ICB/ICB names are title case in Era A and upper case in Era B,
  while `la_rate` names went the other way (upper case in March, title case in June). Names are display strings
  only; never join on them.

**Implication.** Silver needs an organisation dimension keyed on `(org_level, org_code, valid_from)`, plus a
crosswalk table for the 2026-06 ICB reorganisation, and a rule that ONS codes are secondary identifiers.

---
## 9. Data-quality assessment

**Question.** What is actually wrong, what only looks wrong, and what is a methodology change wearing the
costume of a data-quality problem?

Every item below was produced by one of the tests above. The classification is the point: most of these are
*not* defects.

In [34]:
findings = pd.DataFrame([
    ("No exact duplicate rows in any of the 27 CSVs", "expected", "all",
     "Every file also has a working candidate key; the publisher is internally consistent."),
    ("Four distinct non-numeric tokens: '', '*', 'N/A', 'NULL'", "expected", "all",
     "Each has a documented meaning. Must not collapse to NaN."),
    ("Residential type ~14% suppressed in Era A, 0% in Era B", "methodology change", "sicbl_res_type",
     "Era B publishes values 0-4 openly; disclosure control moved to practice merging (note 11)."),
    ("Minimum published value in suppressed Era-A files is 5", "expected", "Era A",
     "Confirms '*' means an integer in 0-4, not 'missing'."),
    ("Measures appear part-way through the Era-A 13-month window", "expected", "several",
     "DELIRIUM_12M from 2025-04; OTHER_(UN)SPECIFIED_DEMENTIA_TYPES from 2025-04; "
     "FRONTOTEMPORAL and LEWY_BODY from 2024-06; MCI absent for 2024-05."),
    ("Cognitive impairment has 12 periods in May 2025, not 13", "expected", "sicbl_cog_imp",
     "Same cause as above - the measure was not published for 2024-05."),
    ("UTLA count steps 132 -> 153 at 2025-07 inside one release", "methodology change", "la_rate",
     "21 county councils plus 2 unitary/metropolitan authorities added. Breaks naive UTLA aggregates."),
    ("42 ICBs become 36 at 2026-06; 23 Sub-ICBs reassigned", "methodology change", "nhs_rate, mapping",
     "Organisational reconfiguration. ICB-level trends cannot cross the boundary."),
    ("England ONS_CODE is 'ENG' in nhs_rate but 'E92000001' elsewhere", "genuine issue", "nhs_rate",
     "Publisher inconsistency. Handle explicitly in the organisation dimension."),
    ("132 ONS codes appear at both LTLA and UTLA", "expected", "la_rate",
     "Unitary authorities are their own upper tier. ORG_TYPE is part of the key."),
    ("GENDER published as Female/Male for DEMENTIA_REGISTER and MCI but FEMALE/MALE for PAT_LIST",
     "genuine issue", "sub_icb_consolidated",
     "Same file, same column, two casings. Breaks GROUP BY GENDER. Dictionary documents only the upper case."),
    ("June mapping file carries a UTF-8 BOM", "genuine issue", "practice_mapping (2026-06)",
     "First column reads as '\\ufeffEXTRACT_DATE' without encoding='utf-8-sig'."),
    ("One June mapping row has literal 'NULL' hierarchy codes", "expected", "practice_mapping (2026-06)",
     "Published sentinel for a practice with no Sub-ICB mapping."),
    ("Dictionary says MEASURE=PAT_LIST_65_PLUS and MEASURE=REVIEW; CSV uses PAT_LIST and REVIEWS",
     "genuine issue", "practice_measures", "Trust the CSV; the dictionary is wrong."),
    ("Dictionary 'pcdem-sub-icb' describes NAME as 'GP practice name'", "genuine issue", "data dictionary",
     "It is the Sub-ICB name. Cosmetic, but a reminder not to trust the dictionary over the data."),
    ("Era-A dictionary documents AGE_SEX_UNKNOWN, which never appears in the data", "needs investigation",
     "sicbl_age_sex", "Documented but absent in both Era-A releases supplied."),
    ("Era-A dictionary documents only SUB_ICB columns for the breakdown files; CSVs carry the full "
     "Region/ICB/Sub-ICB hierarchy", "needs investigation", "Era-A breakdown files",
     "The dictionary lags the data. Twelve columns published, four documented."),
    ("Era-A files pcdem-prac-ass-plans and pcdem-prac-anti-psy are documented but not held locally",
     "needs investigation", "Era A",
     "Practice-level measures exist for Era A but are not in this source set, so the practice time series "
     "currently starts at June 2026."),
    ("Breakdown totals differ from the register total by up to 2 patients per Sub-ICB (Era B)",
     "expected", "sub_icb_consolidated", "Documented in publisher note 2."),
    ("INCIDENCE changes from a 1-month to a 3-month reporting period at 2026-06", "methodology change",
     "INCIDENCE", "Publisher note 7. Values will jump; the series must be broken, not joined."),
    ("REFERRALS introduced in 2026/27; COMORBIDITIES discontinued after 2026-03", "methodology change",
     "sub_icb_consolidated / sicbl_comor_pall_care",
     "Publisher note 6 states the new referral measures are not comparable to previous ones."),
    ("2026-04 and 2026-05 are in no supplied release", "genuine issue", "coverage",
     "Era B publishes one month per release, so those months only exist in their own publications."),
    ("LTLA ONS codes E08000016/E08000019 replaced by E08000038/E08000039 at 2025-08", "methodology change",
     "la_rate", "Count stays at 296; membership changes. Breaks ONS-code joins for those two authorities."),
    ("ICB_NAME in the May 2025 mapping file has leading/trailing spaces on 1.7% of rows", "genuine issue",
     "practice_mapping (2025-05)", "Only whitespace defect in the corpus. Trim names; never join on them."),
    ("Era-A ethnicity/dementia-type/residential-type cover the all-age register; Era-A age/sex covers 65+ only",
     "expected", "Era A",
     "Different denominators inside the same release. Era B puts all four breakdowns on the all-age register."),
], columns=["finding", "classification", "where", "note"])

findings.to_csv(QA / "data_quality_findings.csv", index=False)
findings.groupby("classification").size().to_frame("findings")

,findings
classification,
expected,9
genuine issue,7
methodology change,6
needs investigation,3


In [35]:
findings[["classification", "finding"]].sort_values("classification").reset_index(drop=True)

,classification,finding
0,expected,No exact duplicate rows in any of the 27 CSVs
1,expected,"Four distinct non-numeric tokens: '', '*', 'N/A', 'NULL'"
2,expected,Minimum published value in suppressed Era-A files is 5
3,expected,Measures appear part-way through the Era-A 13-month window
4,expected,"Cognitive impairment has 12 periods in May 2025, not 13"
5,expected,132 ONS codes appear at both LTLA and UTLA
6,expected,One June mapping row has literal 'NULL' hierarchy codes
7,expected,Breakdown totals differ from the register total by up to 2 patient...
8,expected,Era-A ethnicity/dementia-type/residential-type cover the all-age r...
9,genuine issue,England ONS_CODE is 'ENG' in nhs_rate but 'E92000001' elsewhere


**Reading this table.** Only five items are genuine defects, and four of them are cheap to handle in the reader
(BOM, gender casing, England's ONS code, dictionary naming). The expensive ones are the *methodology changes* —
suppression, UTLA expansion, the ICB reorganisation and the incidence redefinition — because no amount of
careful engineering makes those values comparable. They have to be surfaced to the Atlas user, not smoothed
away.

---
## 10. Implications for silver modelling

Design consequences only — nothing here is built yet.

**Fields stable enough to canonicalise**

* `period_end` (`ACH_DATE`) — always a month-end, three spellings across files, parses cleanly.
* `measure` and `breakdown` category *labels* for ethnicity, dementia type and residential type — vocabularies
  are byte-identical across eras.
* ODS organisation codes at Sub-ICB, Region and practice level.
* The five diagnosis-rate measures — identical names, definitions and schema in all three releases.

**Fields requiring a crosswalk**

* `ORG_TYPE` → a controlled `org_level` vocabulary (three source vocabularies for four NHS levels).
* Era-A `Measure` strings → `(measure, breakdown, age, gender)` for `sicbl_age_sex` and `sicbl_cog_imp`;
  → `(measure, breakdown, <dimension>)` for the three category files.
* Era-A `ALL_AGED_<band>` → derive from `Female + Male` rather than storing it as a separate observation
  (proved exactly additive), otherwise silver double-counts on any naive sum.
* ICB codes across the 2026-06 reorganisation — an explicit old→new mapping with effective dates.
* `GENDER` casing inside the June Sub-ICB file.

**Release metadata that must survive transformation**

* `source_release`, `source_file`, `publication_era` (A/B), `ingested_at`.
* `value_raw` alongside `value_num`, plus `value_state` ∈ {numeric, suppressed, blank, not_applicable}.
* `dq_flag` from the rate files (small-denominator warning; only ever set on LTLA/UTLA rows).
* The dictionary version that governs the release (`PCDD-2526` vs `PCDD-2627`).

**Duplicate / revision resolution**

* Evidence says Era-A releases never revise history (100.0000% of 35,302 overlapping observations identical,
  with no coverage differences). A simple "first release that covers the period wins" is currently safe.
* Adopt "latest release wins" anyway and keep `source_release`, because Era B has no overlap to test and a
  re-issued month would otherwise be silently ignored.
* Add a QA assertion that re-runs the section-6 comparison on every new release and fails loudly if a revision
  ever appears.

**Geography complications**

* Organisation dimension keyed `(org_level, org_code, valid_from, valid_to)`; ONS codes as secondary attributes.
* `la_rate` must carry the tier in the key — 132 ONS codes exist at two tiers.
* England needs an explicit alias: `ENG` and `E92000001` are the same entity.
* Flag any ICB-level series that crosses 2026-06; do not interpolate across it.
* Era B publishes Sub-ICB measures at Sub-ICB level only. ICB/Region/England aggregates for those measures were
  published directly in Era A and must now be computed — which means the aggregation rule becomes the Atlas's
  responsibility and should be validated against the Era-A published aggregates before it is trusted.

**Measures that cannot safely be compared across the boundary**

* `INCIDENCE` — reporting period changed from one month to one quarter (note 7).
* `REFERRALS` — new in 2026/27 and explicitly not comparable to earlier referral measures (note 6).
* Residential type counts — 14% suppressed in Era A, unsuppressed in Era B.
* Anything ICB-level spanning 2026-06.
* `COMORBIDITIES` — discontinued; the series simply ends at 2026-03.

**Breakdowns needing special handling**

* Era-A age/sex is 65+ only while Era-A ethnicity/dementia-type/residential-type are all-age. Era B puts all
  four on the same all-age denominator. Store the denominator scope explicitly, per era.
* Suppressed cells must propagate a flag into any aggregate built from them.
* Measure availability is period-dependent; absence of a row is not a zero.

**Ingestion cadence**

* Era A: one release per 12 months back-fills a year.
* Era B: **every** monthly release must be captured, or the month is lost. Back-fill 2026-04 and 2026-05.

---
## 11. Open questions

1. **Are Era-B releases ever revised?** Untestable with one Era-B release. Re-run section 6 as soon as a second
   Era-B month is held that overlaps another publication.
2. **Do the April and May 2026 publications follow Era A or Era B?** The boundary is being attributed to the
   2026/27 financial year, but the exact first Era-B month is inferred, not observed.
3. **Do Era-A published ICB/Region/England aggregates equal the sum of their Sub-ICBs?** If yes, the Era-B
   aggregation rule can be validated against Era A before being trusted. Worth testing directly — the data to do
   it is already loaded.
4. **What happened to the Era-A practice-level files** (`pcdem-prac-ass-plans`, `pcdem-prac-anti-psy`)? They are
   documented but not in this source set, so practice-level history currently begins at June 2026.
5. **Is the Era-B antipsychotic-prescribing breakdown the same measure** as the Era-A practice-level one,
   aggregated to Sub-ICB — or a redefinition?
6. **Why does `AGE_SEX_UNKNOWN` appear in the Era-A dictionary but never in the data?** Retired measure, or a
   category that only materialises in some months?
7. **What is the retention policy for past PCDD publications?** Era B's single-month model makes the Atlas
   dependent on the archive actually being downloadable for every month.
8. **The summary workbooks have not been parsed.** They are presentation tables rather than a modelling source,
   but Table 1–5 would be a useful independent check on any headline figure the Atlas publishes.

---
## Outputs written

All machine-readable QA artefacts are in `outputs/qa/`.

In [36]:
written = sorted(p.name for p in QA.iterdir())
pd.DataFrame({"file": written,
              "size_kb": [round((QA / n).stat().st_size / 1024, 1) for n in written]})

,file,size_kb
0,candidate_keys.csv,4.3
1,column_inventory.csv,31.1
2,data_quality_findings.csv,4.4
3,file_inventory.csv,4.2
4,measure_availability_timeline.csv,9.8
5,measure_crosswalk.csv,3.6
6,measure_inventory.csv,13.6
7,observations_all_releases.parquet,2000.5
8,organisation_levels.csv,2.1
9,release_overlap_matrix.csv,0.1
